## Start GeoFunctions using Spark Esri.

In [51]:
import spark_esri as se  # Make sure this is the first import.

In [52]:
import os
import geofunctions as S
import pyspark.sql.functions as F

### Create a spark instance.

In [53]:
extra_java_options = "-XX:+UseCompressedOops -XX:+AggressiveHeap"

config = {
    "spark.driver.memory": "31G",
    "spark.executor.memory": "31G",
    "spark.driver.extraJavaOptions": extra_java_options,
    "spark.executor.extraJavaOptions": extra_java_options,
    "spark.kryo.unsafe": True,
    "spark.memory.offHeap.enable": True,
    "spark.memory.offHeap.size": "31G",
    "spark.jars": os.path.expanduser("~/geofunctions-0.15.jar"),
}

spark = se.spark_start(config=config)

In [54]:
spark.version

'3.5.3-esri-1'

### Register the spark SQL functions.

In [55]:
S.st_register_functions()

### Test the dataframe functions.

In [23]:
cell = 100.0

In [24]:
(
    spark
    # Generate 10 rows.
    .range(10)
    .withColumn("lon", F.rand() * 360 - 180)
    .withColumn("lat", F.rand() * 180 - 90)
    .withColumn("q", S.st_lontoq("lon", cell))
    .withColumn("r", S.st_lattor("lat", cell))
    .show(truncate=False)
)

+---+-------------------+-------------------+-------+-------+
|id |lon                |lat                |q      |r      |
+---+-------------------+-------------------+-------+-------+
|0  |143.3214854972349  |-33.830491556730166|159544 |-40061 |
|1  |-179.51125574914727|-83.18759976085845 |-199832|-179956|
|2  |33.93497567597049  |-32.96903330013725 |37776  |-38912 |
|3  |57.53126924556568  |-71.28308074824933 |64043  |-114998|
|4  |-14.884447988095332|-68.02363690145886 |-16570 |-104541|
|5  |58.280515100569346 |13.293353157183645 |64877  |14932  |
|6  |38.950099618967215 |15.660020491255224 |43359  |17653  |
|7  |137.4827846307021  |-19.404249424096164|153045 |-22026 |
|8  |-46.115321326904876|39.78290522398078  |-51336 |48344  |
|9  |-165.02292177438494|-56.37080141395634 |-183703|-76326 |
+---+-------------------+-------------------+-------+-------+



### Test the SQL functions.

In [25]:
spark.sql(
    f"""
with t as (
select id,rand() * 360 - 180 lon,rand() * 180 - 90 lat
from range(10)
)

select *,st_lontoq(lon,{cell}D) q, st_lattor(lat,{cell}D) r from t
"""
).show(truncate=False)

+---+-------------------+-------------------+-------+------+
|id |lon                |lat                |q      |r     |
+---+-------------------+-------------------+-------+------+
|0  |-35.72720079349463 |52.19588443350122  |-39772 |68356 |
|1  |164.12748835752365 |-51.761143051070896|182705 |-67571|
|2  |90.94497889083544  |-25.769838995730908|101239 |-29707|
|3  |44.52229789660436  |-26.224169035735343|49561  |-30269|
|4  |-164.30387400716208|-34.876882970889326|-182903|-41472|
|5  |-107.9181901120708 |-41.35307798748606 |-120134|-50646|
|6  |19.075546908662147 |41.709132141977676 |21234  |51175 |
|7  |-87.21293095432763 |41.7863961138178   |-97085 |51290 |
|8  |62.210481283483574 |-7.475067992990361 |69252  |-8345 |
|9  |155.7497966521713  |-32.54614871438769 |173379 |-38353|
+---+-------------------+-------------------+-------+------+

